In [1]:
import numpy as np
import tensorflow as tf

# physical_devices = tf.config.experimental.list_physical_devices('GPU')

# for gpu in physical_devices:
#     tf.config.experimental.set_memory_growth(gpu, True)

from data_loader import EllipticDatasetLoader
from models import GCNTwoLayersSkipConnection
from models.layers import GCNLayer, GCNSkipLayer
from sklearn.metrics import matthews_corrcoef

/home/karim/.local/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
DATADIR = "/home/karim/Projects/Crypto4GraphAI/AML_on_Bitcoin_Elliptic_Dataset/data/elliptic_bitcoin_dataset/"
FILTER_UNKNOWN = False
ONLY_LOCAL_FEATURE = False
CLASS_WEIGTHS = [0.7, 0.29, 0.01]
TEST_SHARE = 0.3
NUM_EPOCH = 500
LEARNING_RATE = 1e-3

In [3]:
def reset_metrics(list_of_metrics):
    for m in list_of_metrics:
        m.reset_states()

In [4]:
dl = EllipticDatasetLoader(DATADIR, TEST_SHARE, FILTER_UNKNOWN, local_features_only=ONLY_LOCAL_FEATURE)

model = GCNTwoLayersSkipConnection(
    layer_gcn=GCNLayer(64,"relu"),
    dropout=tf.keras.layers.Dropout(0.3),
    layer_gcn_skip=GCNSkipLayer(dl.num_classes)
)

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
loss_func = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

2022-11-23 09:44:35.156391: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-11-23 09:44:36.561380: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 150 MB memory:  -> device: 0, name: Quadro RTX 6000, pci bus id: 0000:17:00.0, compute capability: 7.5
2022-11-23 09:44:36.562140: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22180 MB memory:  -> device: 1, name: Quadro RTX 6000, pci bus id: 0000:73:00.0, compute capability: 7.5
2022-11-23 09:44:36.562712: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU

In [5]:
def run_model(adj,nodes,targets,training=False):
    weigths = tf.reduce_sum(CLASS_WEIGTHS * targets, axis=-1)
    logits = model([adj, nodes],training=training)
    loss = loss_func(targets, logits,sample_weight=weigths)
    
    return logits, loss, weigths

In [6]:
train_loss_metric = tf.keras.metrics.Mean()
train_accuracy_metric = tf.keras.metrics.Accuracy()
train_precision_metric = tf.keras.metrics.Precision()
train_recall_metric = tf.keras.metrics.Recall()

test_loss_metric = tf.keras.metrics.Mean()
test_accuracy_metric = tf.keras.metrics.Accuracy()
test_precision_metric = tf.keras.metrics.Precision()
test_recall_metric = tf.keras.metrics.Recall()

metrics = [train_loss_metric, train_accuracy_metric, train_precision_metric, train_recall_metric,
           test_loss_metric, test_accuracy_metric, test_precision_metric, test_recall_metric]

In [7]:
# train_accuracies = []  
# train_precisions = []
# train_recalls = [] 

test_accuracies = []  
test_precisions = []
test_recalls = [] 
test_mcc = [] 

In [8]:
for epoch in range(10):
    reset_metrics(metrics)

    for _, n, t, adj in dl.train_batch_iterator():
        with tf.GradientTape() as tape:
            logits, loss, weigths = run_model(adj, n, t, training=True)

        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads,model.trainable_weights))

        y_true = tf.cast(tf.argmax(t,axis=-1) == 0,tf.float32)
        y_pred = tf.cast(tf.argmax(logits,axis=-1) == 0,tf.float32)
        train_loss_metric(loss)
        train_accuracy_metric(tf.argmax(t,axis=-1), tf.argmax(logits,axis=-1), sample_weight=weigths)
        train_precision_metric(y_true, y_pred)
        train_recall_metric(y_true, y_pred)
        
#         train_accuracies.append(train_accuracy_metric.result().numpy()) 
#         train_precisions.append(train_precision_metric.result().numpy())
#         train_recalls.append(train_recall_metric.result().numpy())

    for _, n, t, adj in dl.test_batch_iterator():
        logits, loss, weigths = run_model(adj, n, t)

        y_true = tf.cast(tf.argmax(t, axis=-1) == 0, tf.float32)
        y_pred = tf.cast(tf.argmax(logits, axis=-1) == 0, tf.float32)
        
        test_loss_metric(loss)
        test_accuracy_metric(tf.argmax(t,axis=-1), tf.argmax(logits,axis=-1), sample_weight=weigths)
        test_precision_metric(y_true, y_pred)                            
        test_recall_metric(y_true, y_pred)
                             
        test_accuracies.append(test_accuracy_metric.result().numpy()) 
        test_precisions.append(test_precision_metric.result().numpy())
        test_recalls.append(test_recall_metric.result().numpy())
        test_mcc.append(matthews_corrcoef(y_true, y_pred))

    print("Epoch: {}\nTRAIN Loss: {:.5}| Accuracy: {:.4}| Precision: {:.4}| Recall: {:.4}".format(
        epoch, train_loss_metric.result().numpy(), train_accuracy_metric.result().numpy(),
        train_precision_metric.result().numpy(),train_recall_metric.result().numpy()
    ))

2022-11-23 09:44:51.056849: E tensorflow/stream_executor/cuda/cuda_blas.cc:232] failed to create cublas handle: CUBLAS_STATUS_NOT_INITIALIZED
2022-11-23 09:44:51.056889: E tensorflow/stream_executor/cuda/cuda_blas.cc:234] Failure to initialize cublas may be due to OOM (cublas needs some free memory when you initialize it, and your deep-learning framework may have preallocated more than its fair share), or may be because this binary was not built with support for the GPU in your machine.
2022-11-23 09:44:51.056910: W tensorflow/core/framework/op_kernel.cc:1745] OP_REQUIRES failed at matmul_op_impl.h:442 : INTERNAL: Attempting to perform BLAS operation using StreamExecutor without BLAS support


InternalError: Exception encountered when calling layer "gcn_layer" (type GCNLayer).

Attempting to perform BLAS operation using StreamExecutor without BLAS support [Op:MatMul]

Call arguments received:
  • inputs=['<tensorflow.python.framework.sparse_tensor.SparseTensor object at 0x7f957b105400>', 'tf.Tensor(shape=(5894, 165), dtype=float32)']
  • training=True
  • mask=None

In [11]:
accuracy = sum(test_accuracies) / len(test_accuracies) 
precision = sum(test_precisions) / len(test_precisions)
recall = sum(test_recalls) / len(test_recalls)
F1 = 2 * (precision * recall) / (precision + recall)
from sklearn.metrics import matthews_corrcoef

In [25]:
accuracy, precision, recall, F1

(0.748220042069753,
 0.08778275611499946,
 0.7775736428101857,
 0.15775594317679387)

In [12]:
accuracy, precision, recall, F1

(0.7228334359765053, 0.07663629953376949, 0.7770969454248746, 0.13951391638554)